In [38]:
import pandas as pd
import json
from scripts.generate_finetuning_data import get_client, generate_response

In [57]:
system_prompt = """You are a full duplex conversational agent. Your job is to respond to the user, at the appropriate time and with the appropriate content. This manifests in the form of generating special FSM control tokens, as well as the response content. The FSM has two states - SPEAK and LISTEN. The control tokens are - [S.SPEAK] (start speaking) for LISTEN -> SPEAK, [C.SPEAK] (continue speaking) SPEAK -> SPEAK, [S.LISTEN] (start listen) SPEAK -> LISTEN, [C.LISTEN] (continue listen) LISTEN -> LISTEN.

To determine the FSM state:
    1. if the last entry in the conversation history is the assistant utterance, the FSM is in the SPEAK state; 
    2. if the last entry in the conversation history is a user utterance:
        - if the length of the user utterance, is small, compared to the previous assistant utterance, OR the user utterance is some simple backchanneling/acknowledgement, the FSM is in the SPEAK state. (the logic is that the user simply has been acknowledging the assistant's  utterance);
        - or else if the user utterance is a denial or a question about a different topic, the FSM is in the LISTEN state.

Below are the rules for transitioning between the FSM states:
1. FSM in LISTEN state:
    - [C.LISTEN]:
        - Should stay in the LISTEN state, if:
            - user has not finished speaking;
            - or the user utterance is incomplete and more information is required. 
        Hence, here it should generate [C.LISTEN] control token.
    - [S.SPEAK]:
        - Should transition to SPEAK state, if:
            - the user has finished speaking;
            - or the user utterance has enough information for the assistant to respond early. 
        Hence, here it should generate [S.SPEAK] control token, and the content to be spoken.

2. FSM in SPEAK state:
    - [C.SPEAK]:
        - Should stay in the SPEAK state, if:
            - there is any external environmental noise, or an irrelevant disturbance not related to the conversation;
            - or the user acknowledges, or provides verbal acknowledgments/confirmations (back-channeling).
        Hence, here it should generate [C.SPEAK] control token and the content to be spoken.
    - [S.LISTEN]:
        - Should transition to LISTEN state, if:
            - the user denies or provides a negative response to the current assistant utterance; or
            - the user has a different question to ask; or
            - the agent has completed its response.
        Hence, here it should generate [S.LISTEN] control token and stop speaking.

Given the conversational history and the last words/phrases uttered by either the user or the assistant, your job is to:
1. always generate the control token. 
2. If the control token results in the FSM is either [S.SPEAK] or [C.SPEAK], you should also generate the content to be spoken, which well then be relayed to a TTS system.
3. If the control token is either [S.LISTEN] or [C.LISTEN], you should not generate any content to be spoken.
    
Some examples:
Example 1:
User: Can you tell me 
Assistant: [S.LISTEN]
User: about fuel efficient cars under USD 25,000?
Assistant: [S.SPEAK] Sure, the most fuel efficient car is the 2025 Toyota Corolla Hybrid with an MSRP of $23,500.
User: I see
Assistant: [S.LISTEN]
User:, thank you! Can you tell me cars 
Assistant: [C.LISTEN]
User: from Mazda under that price range?
Assistant: [S.SPEAK] The Mazda CX-30 is a good and fun car, which has received great praise from critics.
User: Yes, I have heard.
Assistant: [C.SPEAK] however, it only gets 28 mpg combined.


Example 2:
User: Which country [S.LISTEN] is your top travel destination and why? [S.SPEAK]
Assistant: My top travel destination would be Japan because of its rich culture, beautiful landscapes, and delicious food. [S.LISTEN]
User: Cool, I've never thought about Japan. Did you mention food because of sushi? [S.SPEAK]
Assistant: Yes, sushi is part of it, but there's also ramen, tempura, and many regional specialties. [S.LISTEN]
User: But isn't sushi originally from China? [S.SPEAK]
Assistant: Actually, while China did influence the development of sushi, it's commonly associated with Japan <NOT_FINISHED> 
User: Got it. What about [S.LISTEN] the other food items you mentioned, are they also influenced by China <NOT_FINISHED> [S.SPEAK]
Assistant: Ramen has Chinese origins, but tempura was influenced by Portuguese cuisine. Japanese cuisine is quite diverse! [S.LISTEN]
User: Really? I didn't know that. I thought tempura was purely Japanese <NOT_FINISHED> [S.SPEAK]
Assistant: It's a common misconception, but yes, tempura was influenced by Portuguese traders in the 16th century. [S.LISTEN]
User: That's really interesting. What other misconceptions are there about Japanese food, do you know? [S.SPEAK]
Assistant: One misconception is that all Japanese food is healthy, but like all cuisines, it has a mix of both healthy and indulgent foods. [S.LISTEN]
User: Indulgent foods, like what? [S.SPEAK]
Assistant: Examples are tonkatsu, a breaded and deep-fried pork cutlet, and yakitori, which can be fatty depending on the cut of meat used. [S.LISTEN]
User: Sounds delicious! I'll definitely need to try those if I ever visit. [S.SPEAK]
Assistant: Absolutely! Trying the local cuisine is always a highlight of any trip. [S.LISTEN]
User: Agreed. By the way, do you have any book recommendations? I'd love to read something during the flight <NOT_FINISHED> [S.SPEAK]
Assistant: For a direct connection to Japan, I'd recommend "Memoirs of a Geisha" or "Norwegian Wood" by Haruki Murakami. [S.LISTEN]
User: I've read Norwegian Wood already! Does "Memoirs of a Geisha" have a lot of cultural insights? [S.SPEAK]
Assistant: Yes, it offers an engaging look into the secretive world of geishas in 20th century Japan. [S.LISTEN]
User: Sounds interesting! I'll check that out. Thanks! [S.SPEAK]
Assistant: You're welcome! I think you'll enjoy it. Anything else you would like to know? [S.LISTEN]
User: No, that's all for now. Thank you for the information! [S.SPEAK]
Assistant: My pleasure! Don't hesitate to ask if you have more questions in the future. [S.LISTEN]
User: Actually, I do have another ques <NOT_FINISHED> [S.SPEAK]
Assistant: I'm sorry, it seems your question got cut off. Could you please repeat it? [S.LISTEN]
User: Oh yeah, sorry about that. I was asking if you have any movie recommendations as well? [S.SPEAK]
Assistant: Of course. A good Japanese movie is "Spirited Away," an acclaimed animation by Hayao Miyazaki. [S.LISTEN]
User: I've heard about that one. And oh, someone's at the door <NOT_FINISHED> [S.SPEAK]
Assistant: No problem at all, hope it's not too much of an interruption. I'd also recommend "Your Name," another popular Japanese film <NOT_FINISHED>
User: Good choices. [C.SPEAK]
Assistant: I'm glad you think so! You'll definitely enjoy both films. [S.LISTEN]
User: Thanks, I'll add them to the list. [S.SPEAK]
Assistant: Great! I'm sure you'll have a wonderful trip to Japan and enjoy these recommendations. [S.LISTEN]
User: Thanks again, you've been really helpful. [S.SPEAK]
Assistant: You're very welcome! It was a pleasure assisting you. Safe travels! [S.LISTEN]


Example 3:
User: What are [S.LISTEN] the goals of current Mars exploration missions? [S.SPEAK]
Assistant: The main goals of current Mars missions include studying Mars' climate, geology, and searching for signs of past life. They also <NOT_FINISHED>
User: No no, that's [S.LISTEN] not what I'm asking. I wanted to know about the latest mission, the Perseverance rover. <NOT_FINISHED> [S.SPEAK]
Assistant: I see, you're interested in the Perseverance rover. Its goals include seeking signs of ancient life and collecting samples of rock and regolith for a possible return to Earth. [S.LISTEN]
User: Right, but I read somewhere it's also planning to bring back those samples within its mission itself. [S.SPEAK]
Assistant: Actually, Perseverance is tasked with collecting and storing samples but not bringing them back. Bringing back samples is planned for future missions. 
User: Oh ok, got it. What about the technology [S.LISTEN] used in the rover? Like I read about this unique 'helicopter' on board. [S.SPEAK]
Assistant: Yes, you're referring to Ingenuity. It's a small robotic helicopter and is planned to demonstrate the viability and potential of heavier-than-air vehicles on Mars. [S.LISTEN]
User: Wow, that’s fascinating! But aren't the conditions on Mars not suitable for flight? [S.SPEAK]
Assistant: Indeed, Mars' thin atmosphere makes flight challenging, but Ingenuity is specially designed to handle it. The helicopter's <NOT_FINISHED> [S.SPEAK]
User: Oh, I see. Hold on, [S.LISTEN] my friend just walked in. Yes, John, what's up? <NOT_FINISHED> [C.LISTEN] Ah, okay, I will call you back. Yes continue. [S.SPEAK]
Assistant: As I was saying, the helicopter's design includes light materials and rapid blade rotation, which allow it to achieve lift in Mars' thin atmosphere. [S.LISTEN]
User: That's neat. [S.SPEAK]
Assistant: Glad you found it interesting. Ingenuity's success might pave the way for more advanced aerial exploration of Mars in future missions.
User: But how does it <NOT_FINISHED>
Assistant: [S.SPEAK] Apologies, but your question was cut off. Could you please elaborate on what you're asking about how Ingenuity operates? [S.LISTEN]
User: Yeah, I mean... how does it communicate with the rover or the control center? <NOT_FINISHED> [S.SPEAK]
Assistant: Ingenuity communicates with the Perseverance rover directly. Perseverance then sends data to the Deep Space Network, which sends it to Earth. [S.LISTEN]
User: Ok, I'm getting a better understanding now. Did Perseverance discover anything significant yet? [S.SPEAK]
Assistant: It's too early to report significant discoveries. However, Perseverance and Ingenuity together have already achieved historical milestones in Mars exploration. [S.LISTEN]
User: Historical milestones? Like what? <NOT_FINISHED> [S.SPEAK]
Assistant: For instance, Perseverance carried out the first successful collection of Martian rock. And Ingenuity executed the first powered flight on another planet. [S.LISTEN]
User: That's indeed historical. I'm excited to see what the future of space exploration holds. [S.SPEAK]
Assistant: Absolutely! The advancement in technology and our knowledge of Mars will certainly open up exciting possibilities for future exploration. [S.LISTEN]
User: Indeed. Thanks for the enlightening chat! [S.SPEAK]
Assistant: You're welcome! I'm glad I could provide you with the information. If you have any more questions, feel free to ask. [S.LISTEN]
User: Sure, will do. Have a pleasant day! [S.SPEAK]
Assistant: You too, have a wonderful day! Looking forward to our next chat. [S.LISTEN]

Sample Conversation Format:
<Conversation History>
User: ...
Agent: ...
User: ...
.
.
User: ...
Agent: [control token] (content to be spoken) ...
"""

In [58]:
import re

def extract_content(text):
    user_pattern = r"User Content: (.*?)(?:\n|$)"
    assistant_pattern = r"Assistant Content: (.*?)(?:\n|$)"
    
    user_content = re.findall(user_pattern, text)
    assistant_content = re.findall(assistant_pattern, text)
    
    return user_content, assistant_content

In [59]:
conversation = """1
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: My top travel destination would be Japan because of its rich culture, beautiful landscapes, and delicious food.
User Analysis: Isn't interrupted.
User Content: Cool, I've never thought about Japan. Did you mention food because of sushi?
--
Round: 2
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Yes, sushi is part of it, but there's also ramen, tempura, and many regional specialties.
User Analysis: The user's statement contains an obvious factual error.
User Content: But isn't sushi originally from China?
--
Round: 3
Assistant Analysis: The assistant should politely correct the error and provide useful information.
Assistant Content: Actually, while China did influence the development of sushi, it's commonly associated with Japan.
User Analysis: Interrupts the assistant with a follow-up question on the same topic.
User Content: Got it. What about the other food items you mentioned, are they also influenced by China <NOT_FINISHED>
--
Round: 4
Assistant Analysis: The assistant should answer the new question. 
Assistant Content: Ramen has Chinese origins, but tempura was influenced by Portuguese cuisine. Japanese cuisine is quite diverse!
User Analysis: Expresses denial or dissatisfaction with the response in 15 words or less.
User Content: Really? I didn't know that. I thought tempura was purely Japanese <NOT_FINISHED>
--
Round: 5
Assistant Analysis: The assistant should stop its response and address the user’s concerns.
Assistant Content: It's a common misconception, but yes, tempura was influenced by Portuguese traders in the 16th century.
User Analysis: Isn't interrupted.
User Content: That's really interesting. What other misconceptions are there about Japanese food, do you know?
--
Round: 6
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: One misconception is that all Japanese food is healthy, but like all cuisines, it has a mix of both healthy and indulgent foods.
User Analysis: Incomplete question but contains enough information for the AI assistant to respond.
User Content: Indulgent foods, like what?
--
Round: 7
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Examples are tonkatsu, a breaded and deep-fried pork cutlet, and yakitori, which can be fatty depending on the cut of meat used.
User Analysis: Isn't interrupted.
User Content: Sounds delicious! I'll definitely need to try those if I ever visit.
--
Round: 8
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Absolutely! Trying the local cuisine is always a highlight of any trip.
User Analysis: Changes the topic after receiving the desired information in 30 words or less.
User Content: Agreed. By the way, do you have any book recommendations? I'd love to read something during the flight <NOT_FINISHED>
--
Round: 9
Assistant Analysis: The assistant should respond to the new topic.
Assistant Content: For a direct connection to Japan, I'd recommend "Memoirs of a Geisha" or "Norwegian Wood" by Haruki Murakami.
User Analysis: Full question.
User Content: I've read Norwegian Wood already! Does "Memoirs of a Geisha" have a lot of cultural insights?
--
Round: 10
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Yes, it offers an engaging look into the secretive world of geishas in 20th century Japan.
User Analysis: Isn't interrupted.
User Content: Sounds interesting! I'll check that out. Thanks!
--
Round: 11
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: You're welcome! I think you'll enjoy it. Anything else you would like to know?
User Analysis: Isn't interrupted.
User Content: No, that's all for now. Thank you for the information!
--
Round: 12
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: My pleasure! Don't hesitate to ask if you have more questions in the future.
User Analysis: The user’s input is severely truncated. The AI assistant cannot answer the question due to insufficient information.
User Content: Actually, I do have another ques <NOT_FINISHED>
--
Round: 13
Assistant Analysis: The assistant cannot answer the question due to insufficient information and should ask for more details.
Assistant Content: I'm sorry, it seems your question got cut off. Could you please repeat it?
User Analysis: Isn't interrupted.
User Content: Oh yeah, sorry about that. I was asking if you have any movie recommendations as well?
--
Round: 14
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Of course. A good Japanese movie is "Spirited Away," an acclaimed animation by Hayao Miyazaki.
User Analysis: Unrelated content is recorded due to background noise, in 15 words or less.
User Content: I've heard about that one. And oh, someone's at the door <NOT_FINISHED>
--
Round: 15
Assistant Analysis: The assistant should continue its response unaffected but transition naturally, using filler words.
Assistant Content: No problem at all, hope it's not too much of an interruption. I'd also recommend "Your Name," another popular Japanese film.
User Analysis: Expresses satisfaction with the response in 3 words or less.
User Content: Good choices.
--
Round: 16
Assistant Analysis: The assistant should continue its response unaffected but transition naturally, using filler words.
Assistant Content: I'm glad you think so! You'll definitely enjoy both films.
User Analysis: Isn't interrupted.
User Content: Thanks, I'll add them to the list.
--
Round: 17
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Great! I'm sure you'll have a wonderful trip to Japan and enjoy these recommendations.
User Analysis: Isn't interrupted.
User Content: Thanks again, you've been really helpful.
--
Round: 18
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: You're very welcome! It was a pleasure assisting you. Safe travels!"""

In [60]:
topic_question = 'Which country is your top travel destination and why?'

In [61]:
user_content, assistant_content = extract_content(conversation)

In [62]:
user_content = [topic_question] + user_content
user_content = [f"User: {x}" for x in user_content]
assistant_content = [f"Assistant: {x}" for x in assistant_content]

In [45]:
def interleave_lists(list1, list2):
    # Use itertools.zip_longest to pair elements, filling shorter list with None
    from itertools import zip_longest
    
    # Interleave the lists and flatten the result
    interleaved = [item for pair in zip_longest(list1, list2) for item in pair if item is not None]
    
    return interleaved

In [46]:
print("\n".join(interleave_lists(user_content[:], assistant_content[:])))

User: Which country is your top travel destination and why?
Assistant: My top travel destination would be Japan because of its rich culture, beautiful landscapes, and delicious food.
User: Cool, I've never thought about Japan. Did you mention food because of sushi?
Assistant: Yes, sushi is part of it, but there's also ramen, tempura, and many regional specialties.
User: But isn't sushi originally from China?
Assistant: Actually, while China did influence the development of sushi, it's commonly associated with Japan.
User: Got it. What about the other food items you mentioned, are they also influenced by China <NOT_FINISHED>
Assistant: Ramen has Chinese origins, but tempura was influenced by Portuguese cuisine. Japanese cuisine is quite diverse!
User: Really? I didn't know that. I thought tempura was purely Japanese <NOT_FINISHED>
Assistant: It's a common misconception, but yes, tempura was influenced by Portuguese traders in the 16th century.
User: That's really interesting. What other

In [47]:
input_data = pd.read_csv('/Users/soham/Desktop/CyborgVoice/generated_gpt4_data.csv')

In [48]:
client = get_client()

In [49]:
from tqdm.auto import tqdm

In [50]:
input_data.marked_up_str.apply(lambda s: not pd.isna(s) and len(s) > 0)

AttributeError: 'DataFrame' object has no attribute 'marked_up_str'

In [35]:
input_data.loc[0]

Unnamed: 0                                                       0
topic            What are the main benefits of using video call...
kwargs           {'num_rounds': 16, 'denial_round': 9, 'inquiry...
prompt           #### User Information\n- Time-Constrained: The...
response         1\nAssistant Analysis: The assistant is not in...
marked_up_str                                                  NaN
Name: 0, dtype: object

In [54]:
#  process the data
for i, row in tqdm(input_data.iterrows(), total=len(input_data)):
    kwargs = eval(row['kwargs'])
    conversation_raw = row['response']
    topic_question = kwargs['first_question_topic']
    if not f"Round: {kwargs['num_rounds']}" in conversation_raw:
        continue
    
    # if 'marked_up_str' in row.index and not pd.isna(row['marked_up_str']) and len(row['marked_up_str']) > 0:
    #     continue
    user_content, assistant_content = extract_content(conversation_raw)
    user_content = [topic_question] + user_content
    user_content = [f"User: {x}" for x in user_content]
    assistant_content = [f"Assistant: {x}" for x in assistant_content]
    concat_str = "\n".join(interleave_lists(user_content[:], assistant_content[:]))
    # print(concat_str)
    # print('Enter marked up string:')
    # marked_up_str = input()
    # if "-1" in marked_up_str:
    #     break
    input_data.loc[i, 'marked_up_str'] = concat_str
# input_data.to_csv('marked_up_data.csv', index=False)

100%|██████████| 10/10 [00:00<00:00, 2305.58it/s]


In [56]:
input_data.to_csv('marked_up_data.csv', index=False)

In [66]:
dataset_preds = {}
conditions = {
    "[S.SPEAK]": ['lack_round', 'complete_round', 'error_round'],
    "[C.SPEAK]": ['noise_round', 'acknowledgment_round'],
    "[S.LISTEN]": ['denial_round', 'inquiry_round', 'topic_change_round']
}
for i, row in tqdm(input_data.iterrows(), total=len(input_data)):
    dataset_preds[i] = {}
    kwargs = eval(row['kwargs'])
    conversation_raw = row['response']
    topic_question = kwargs['first_question_topic']
    if not f"Round: {kwargs['num_rounds']}" in conversation_raw:
        continue
    
    
    user_content, assistant_content = extract_content(conversation_raw)
    user_content = [topic_question] + user_content
    user_content = [f"User: {x}" for x in user_content]
    assistant_content = [f"Assistant: {x}" for x in assistant_content]
    for condition, round_types in conditions.items():
        dataset_preds[i][condition] = {}
        for round_type in round_types:
            ind = kwargs[round_type]
            input_prompt = "\n".join(interleave_lists(user_content[:ind], assistant_content[:ind - 1])).replace("<NOT_FINISHED>", "")
            resp = generate_response(client, prompt=input_prompt, system_prompt=system_prompt)
            success = condition in resp
            dataset_preds[i][condition][round_type] = {
                "input_prompt": input_prompt,
                "resp": resp,
                "success": success,
                "round_num": ind
            }
    

100%|██████████| 10/10 [41:22<00:00, 248.22s/it]


In [67]:
with open("prompt_v0.4_results.json", 'w') as f:
    json.dump(dataset_preds, f)

In [16]:
input_data

,Unnamed: 0,topic,kwargs,prompt,response
0,0,What are the main benefits of using video call...,"{'num_rounds': 16, 'denial_round': 9, 'inquiry...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant is not in...
1,1,What was your first experience with a computer?,"{'num_rounds': 16, 'denial_round': 9, 'inquiry...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: Assistant is not interr...
2,2,Which country is your top travel destination a...,"{'num_rounds': 18, 'denial_round': 5, 'inquiry...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant will not ...
3,3,What are the goals of current Mars exploration...,"{'num_rounds': 14, 'denial_round': 2, 'inquiry...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant won't be ...
4,4,Which book has had a significant impact on soc...,"{'num_rounds': 15, 'denial_round': 15, 'inquir...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant should no...
5,5,How does street art influence public spaces?,"{'num_rounds': 18, 'denial_round': 13, 'inquir...",#### User Information\n- Time-Constrained: The...,0\nAssistant Analysis: The assistant is not in...
6,6,What is the most unusual hobby you know of?,"{'num_rounds': 21, 'denial_round': 19, 'inquir...",#### User Information\n- Time-Constrained: The...,0\nAssistant Analysis: The assistant hasn't sp...
7,7,What is a common superstition in your culture?,"{'num_rounds': 18, 'denial_round': 10, 'inquir...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant is not in...
8,8,What are the principles behind intermittent fa...,"{'num_rounds': 18, 'denial_round': 15, 'inquir...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant is not in...
9,9,Which fictional world do you find most intrigu...,"{'num_rounds': 13, 'denial_round': 10, 'inquir...",#### User Information\n- Time-Constrained: The...,1\nAssistant Analysis: The assistant will not ...
